## **SIN 393 – Introduction to Computer Vision (2026)**

### Prof. João Fernando Mari ([*joaofmari.github.io*](https://joaofmari.github.io/))

Copyright (c) 2026 João Fernando Mari.  
Licensed under the MIT License.

---

# Lecture 3 - Notebook 2 - Perceptron: Decision Boundary and Error Surface
---

## Importing the Required Libraries


In [11]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

## Computing the Error Surface

For a Perceptron with two inputs, the model parameters are

$
w_1,\quad w_2,\quad b.
$

For each combination of these parameters, the Perceptron prediction is

$
\hat{y} =
\begin{cases}
1, & w_1x_1 + w_2x_2 + b \geq 0 \\
0, & \text{otherwise}.
\end{cases}
$

We evaluate a discrete grid of values for $w_1$, $w_2$, and $b$.

The total error is

$
E = \sum_i (y_i-\hat{y}_i)^2.
$

Since $y_i,\hat{y}_i \in \{0,1\}$, this error corresponds to the number of misclassified samples.

In [12]:
def error_brute_force(
    X, y,
    w1_range=(-1, 1),
    w2_range=(-1, 1),
    b_range=(-1, 1),
    step=0.1
):
    """
    Computes the classification error for all combinations
    of w1, w2, and b in a discrete parameter grid.
    """

    # Parameter values.
    w1_values = np.arange(w1_range[0], w1_range[1], step)
    w2_values = np.arange(w2_range[0], w2_range[1], step)
    b_values = np.arange(b_range[0], b_range[1], step)

    # Initialize the error surface.
    error_surface = np.zeros((
        len(w1_values),
        len(w2_values),
        len(b_values)
    ))

    # Test all parameter combinations.
    for i, w1 in enumerate(w1_values):
        for j, w2 in enumerate(w2_values):
            for k, b in enumerate(b_values):

                w = np.array([w1, w2])

                # Weighted sum.
                v = X @ w + b

                # Activation function.
                y_hat = (v >= 0).astype(int)

                # Error.
                e = y - y_hat

                # Total error.
                error_surface[i, j, k] = np.sum(e**2)

    return error_surface, w1_values, w2_values, b_values

## Plotting the Decision Boundary and Error Surface

The Perceptron has three parameters:

$
(w_1,w_2,b).
$

Therefore, its error is a function of three variables:

$
E(w_1,w_2,b).
$

To visualize this error surface, we fix one parameter and plot a 2D slice of the remaining two parameters.

In [13]:
def plot_decision_boundary(ax, X, y, w1, w2, b, title):
    """
    Plots the samples and the Perceptron decision boundary.
    """

    colors = ['r', 'g', 'b', 'y', 'c', 'm']

    # Plot the samples.
    for y_ in np.unique(y):
        ax.scatter(
            X[y == y_, 0],
            X[y == y_, 1],
            color=colors[y_],
            label=f'Class {y_}',
            zorder=3
        )

    # Plot the decision boundary:
    # w1*x1 + w2*x2 + b = 0
    if w2 != 0:
        x1 = np.array([
            X[:, 0].min() - 1.,
            X[:, 0].max() + 1.
        ])

        x2 = -(w1 / w2) * x1 - (b / w2)

        ax.plot(x1, x2, color='k', zorder=2)

    elif w1 != 0:
        # Vertical decision boundary.
        x1 = -b / w1
        ax.axvline(x=x1, color='k', zorder=2)

    # Labels and title.
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(title)
    ax.legend()

    # Visualization limits.
    ax.set_xlim(
        X[:, 0].min() - .5,
        X[:, 0].max() + .5
    )
    ax.set_ylim(
        X[:, 1].min() - .5,
        X[:, 1].max() + .5
    )

    # Coordinate axes and grid.
    ax.axhline(y=0, color='gray', linewidth=1)
    ax.axvline(x=0, color='gray', linewidth=1)

    ax.grid(
        color='lightgray',
        linestyle='--',
        linewidth=0.5
    )
    ax.set_axisbelow(True)

In [14]:
def plot_error_slice(
    ax,
    x_values,
    y_values,
    error_slice,
    x_current,
    y_current,
    x_label,
    y_label,
    title,
    surface=True,
    vmin=None,
    vmax=None
):
    """
    Plots one 2D slice of the error surface.
    """

    # Current point.
    i = np.argmin(np.abs(x_values - x_current))
    j = np.argmin(np.abs(y_values - y_current))

    error_current = error_slice[i, j]

    if surface:
        # Parameter grid.
        X_grid, Y_grid = np.meshgrid(
            x_values,
            y_values,
            indexing='ij'
        )

        # Error surface.
        ax.plot_surface(
            X_grid,
            Y_grid,
            error_slice,
            alpha=0.8
        )

        # Vertical locator line.
        ax.plot(
            [x_current, x_current],
            [y_current, y_current],
            [0, error_slice.max()],
            color='r',
            linewidth=2
        )

        # Current error point.
        ax.scatter(
            x_current,
            y_current,
            error_current,
            color='r',
            s=60,
            depthshade=False
        )

        ax.set_zlabel('Error')

    else:
        # Error as a grayscale image.
        im = ax.imshow(
            error_slice.T,
            origin='lower',
            extent=[
                x_values[0],
                x_values[-1],
                y_values[0],
                y_values[-1]
            ],
            aspect='auto',
            cmap='gray',
            vmin=vmin,
            vmax=vmax
        )

        # Current parameter combination.
        ax.plot(
            x_current,
            y_current,
            'ro',
            markersize=7
        )

        # Colorbar for this plot.
        ax.figure.colorbar(
            im,
            ax=ax,
            label='Error'
        )

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)

In [15]:
def plot_error_surface(
    X,
    y,
    title,
    error_surface,
    i_w1,
    i_w2,
    i_b,
    w1_values,
    w2_values,
    b_values,
    surface=True
):
    """
    Plots the decision boundary and the error surface.
    """

    # Current parameter values.
    w1 = w1_values[i_w1]
    w2 = w2_values[i_w2]
    b = b_values[i_b]

    error = error_surface[i_w1, i_w2, i_b]

    print(
        f'w1: {w1:.2f}  '
        f'w2: {w2:.2f}  '
        f'b: {b:.2f}  '
        f'Error: {error:.2f}'
    )

    if not surface:
        vmin = error_surface.min()
        vmax = error_surface.max()
    else:
        vmin = None
        vmax = None

    fig = plt.figure(figsize=(12, 7))

    # ---------------------------------------------
    # Decision boundary
    # ---------------------------------------------

    ax1 = fig.add_subplot(2, 3, 1)

    plot_decision_boundary(
        ax1,
        X,
        y,
        w1,
        w2,
        b,
        title
    )

    # ---------------------------------------------
    # Error: w1 x w2
    # Fixed: b
    # ---------------------------------------------

    ax2 = fig.add_subplot(
        2, 3, 4,
        projection='3d' if surface else None
    )

    plot_error_slice(
        ax2,
        w1_values,
        w2_values,
        error_surface[:, :, i_b],
        w1,
        w2,
        '$w_1$',
        '$w_2$',
        f'$b={b:.2f}$',
        surface,
        vmin,
        vmax
    )

    # ---------------------------------------------
    # Error: w1 x b
    # Fixed: w2
    # ---------------------------------------------

    ax3 = fig.add_subplot(
        2, 3, 5,
        projection='3d' if surface else None
    )

    plot_error_slice(
        ax3,
        w1_values,
        b_values,
        error_surface[:, i_w2, :],
        w1,
        b,
        '$w_1$',
        '$b$',
        f'$w_2={w2:.2f}$',
        surface,
        vmin,
        vmax
    )

    # ---------------------------------------------
    # Error: w2 x b
    # Fixed: w1
    # ---------------------------------------------

    ax4 = fig.add_subplot(
        2, 3, 6,
        projection='3d' if surface else None
    )

    plot_error_slice(
        ax4,
        w2_values,
        b_values,
        error_surface[i_w1, :, :],
        w2,
        b,
        '$w_2$',
        '$b$',
        f'$w_1={w1:.2f}$',
        surface,
        vmin,
        vmax
    )

    plt.tight_layout()
    plt.show()

## Defining Datasets: Binary Logic Functions

We use the four possible combinations of two binary inputs:

$
(x_1,x_2) \in \{(0,0),(0,1),(1,0),(1,1)\}.
$

The corresponding labels are defined for the AND, OR, and XOR functions.

In [16]:
# Binary input data
X_bin = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

# Binary logic functions
# ======================

# AND
y_and = np.array([0, 0, 0, 1])

# OR
y_or = np.array([0, 1, 1, 1])

# XOR - Not linearly separable
y_xor = np.array([0, 1, 1, 0])

### Select a Logic Function

In [17]:
X = X_bin
y = y_and
title = 'AND'

## Computing the Error Surface

Evaluate all parameter combinations in the predefined grid.

In [18]:
error_surface, w1_values, w2_values, b_values = error_brute_force(X, y)
print(f'Error surface shape: {error_surface.shape}')

Error surface shape: (20, 20, 20)


## Random Parameter Initialization

Select an initial combination of $$w_1$, $w_2$, and $b$ from the parameter grid.

In [19]:
# Random parameter initialization.
i_w1 = np.random.randint(len(w1_values))
i_w2 = np.random.randint(len(w2_values))
i_b = np.random.randint(len(b_values))

print(f'w1: {w1_values[i_w1]:.1f}')
print(f'w2: {w2_values[i_w2]:.1f}')
print(f'b:  {b_values[i_b]:.1f}')

w1: 0.5
w2: -0.9
b:  -0.9


## Exploring the Decision Boundary and Error Surface

Use the sliders to change \(w_1\), \(w_2\), and \(b\).

Observe how:

- the parameters change the decision boundary;
- the classification error changes;
- the selected parameter combination moves through the error surface.

Use **3D Surface** or **Gray Image** to visualize the error-surface slices.

In [20]:
slider_w1 = widgets.SelectionSlider(
    options=[(f'{value:.1f}', i) for i, value in enumerate(w1_values)],
    style={'description_width': '30px'},
    value=i_w1,
    description='w1'
)

slider_w2 = widgets.SelectionSlider(
    options=[(f'{value:.1f}', i) for i, value in enumerate(w2_values)],
    style={'description_width': '30px'},
    value=i_w2,
    description='w2'
)

slider_b = widgets.SelectionSlider(
    options=[(f'{value:.1f}', i) for i, value in enumerate(b_values)],
    style={'description_width': '30px'},
    value=i_b,
    description='b'
)

widgets.interact(
    plot_error_surface,
    X=widgets.fixed(X),
    y=widgets.fixed(y),
    title=widgets.fixed(title),
    error_surface=widgets.fixed(error_surface),
    i_w1=slider_w1,
    i_w2=slider_w2,
    i_b=slider_b,
    w1_values=widgets.fixed(w1_values),
    w2_values=widgets.fixed(w2_values),
    b_values=widgets.fixed(b_values),
    surface=widgets.ToggleButtons(
        options=[
            ('3D Surface', True),
            ('Gray Image', False)
        ],
        value=True,
        description='Plot:'
    )
)

interactive(children=(SelectionSlider(description='w1', index=15, options=(('-1.0', 0), ('-0.9', 1), ('-0.8', …

<function __main__.plot_error_surface(X, y, title, error_surface, i_w1, i_w2, i_b, w1_values, w2_values, b_values, surface=True)>

Repeat the experiment using:

- AND
- OR
- XOR

## License
---
Copyright (c) 2026 João Fernando Mari.

The Python source code and Jupyter Notebook are licensed under the MIT License.  
See the `LICENSE` file in the repository for details.

Third-party materials remain subject to the rights and licenses of their respective authors and owners.

## Bibliography
---

* GONZALEZ, R.C.; WOODS, R.E.; Digital Image Processing. 3rd Edition. Prentice Hall, 2008.
* DUDA, R.O.; HART, P.E.; STORK, D.G. Pattern Classification. 2nd Edition. Wiley-Interscience, 2001.
* COSTA, L. DA F.; CESAR-JR., R. M. Shape analysis and classification: theory and practice. CRC Press, 2000. Chapter 8.
* ROSENBLATT, F. The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain. Psychological Review, 65(6), 386–408, 1958.
* scikit-learn - User Guide.
    * https://scikit-learn.org/stable/user_guide.html